2026.10.28 一致性修订：请 Restart Kernel & Run All。初始化会验证当前 embedding→特征→预测→概率；旧 niche 缓存不再复用，旧图输出已清空。Deployment 最佳模型用法保持不变。


# BRCA external validation — pan-cancer Hist2Pheno (UNI2 A2)

UNI2 A2 uses a 14 px crop (7 µm after rescale to 0.5 µm/px). Run `demo_uni2_A2.sh`. UNI predictions stay in `pred_pancancer_section`.


Three HE + StarDist slides under `data/Visium/BRCA` (**no cell-type GT**):

| Sample | HE | µm/px |
|--------|----|-------|
| `GSM7782699_tissue_image` | OME TIFF | ~0.548 (OME PhysicalSize) |
| `200715_BC_SA2_C_D_V19F24-113.B1-Spot000001` | pathology JPEG → Visium `V10F24-113_B1` | ~0.172 (Visium × HE÷SR) |
| `200219_BC_SA3_V19T26-031_CE_V4-Spot000001` | pathology JPEG → Visium `V19T26-031_D1` | ~0.172 (Visium × HE÷SR) |

Predictions use the pan-cancer **cross-section** checkpoint
`A3_adjacent_spatial_k8mean_gate` (L1 / L12 / L2), spatial k=8.
Cohort helper: `code/External_val/BRCA/brca_external.py` (mixed HE suffix + per-sample mpp).
Zenodo JPEGs are ~3.333× Space Ranger fullres — do **not** use raw `65/spot_diameter` on them.

```bash
conda activate SeededNTM
bash code/External_val/BRCA/demo_uni2_A2.sh dry-run
SAMPLE=200715_BC_SA2_C_D_V19F24-113.B1-Spot000001 bash code/External_val/BRCA/demo_uni2_A2.sh all
```

Outputs: `data/Visium/BRCA/pred_uni2_A2_bestmlp_paired_v2_seed42/`.


In [ ]:
import sys
from pathlib import Path

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
# 2026.10.28, explicit checkout override / Farm 可设 HIST2PHENO_REPO
import os
if os.environ.get("HIST2PHENO_REPO"):
    REPO = Path(os.environ["HIST2PHENO_REPO"]).expanduser().resolve()
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
import os
os.environ["UNI2_CKPT"] = str(REPO / "data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt")
STAR_PROB_MIN = 0.0  # 2026.10.28, set above 0 to store a separate _probXX folder
import os as _prob_os
_prob_os.environ["STAR_PROB_MIN"] = "0" if float(STAR_PROB_MIN) <= 0 else f"{float(STAR_PROB_MIN):.2f}"
os.environ["UNI2_PRED_DIRNAME"] = "pred_uni2_A2_bestmlp_paired_v2_seed42"
sys.path.insert(0, str(REPO / "code"))
sys.path.insert(0, str(REPO / "code" / "External_val" / "BRCA"))

import brca_external as brca_ext
brca_ext.use_embedder("uni2")

PRED_ROOT = brca_ext.PRED_ROOT
composition_table = brca_ext.composition_table
discover_samples = brca_ext.discover_samples
geometry = brca_ext.geometry
sample_paths = brca_ext.sample_paths
ensure_l2_probs = brca_ext.ensure_l2_probs
plot_spatial = brca_ext.plot_spatial
plot_composition = brca_ext.plot_composition
plot_roi_spatial = brca_ext.plot_roi_spatial
plot_roi_composition = brca_ext.plot_roi_composition
# Re-run the plot cell after changing these. Notebook display uses DPI; saved PNGs use SAVE_DPI.
SAVE_FIGURES = False
FIGSIZE = (20.4, 7.8)
DPI = 200
SAVE_DPI = 300
HE_MAX_SIDE = 1400
MARKER_PX = 1.0

samples = discover_samples()
for sample in samples:
    g = geometry(sample)
    print(
        f"{sample}  mpp={g['mpp_um']:.4f} µm/px  "
        f"({g['mpp_source']})  "
        f"native14={g['patch_um_if_native_16px']:.2f} µm  "
        f"after rescale={g['patch_um_after_rescale']:.1f} µm  "
        f"scale={g['scale']:.4f}"
    )
    p = sample_paths(sample)
    print(f"  HE: {p['he'].name}  pred exists: {p['pred'].is_file()}")

# 2026.10.28, mandatory current-input preflight / 后续直接读 h5ad 的图表也不得绕过上游
import os as _pipeline_os
_pipeline_os.environ["STAR_PROB_MIN"] = str(globals().get("STAR_PROB_MIN", 0.0))
if "ensure_l2_probs" in globals():
    _owner = ensure_l2_probs.__self__
    _current_samples = list(samples) if "samples" in globals() else _owner.discover_samples()
    _owner.cmd_predict(_current_samples, force=False, allow_cpu=True)
elif "ue" in globals():
    ue.cmd_predict(COHORT, samples, cuda_device=_pipeline_os.environ.get("CUDA_DEVICE", "0"),
                   force=False, allow_cpu=True, embedder=EMBEDDER)
else:
    raise RuntimeError("No current prediction validator; run the setup cell first")


In [ ]:
import pandas as pd
from IPython.display import Image, display

missing = [s for s in samples if not sample_paths(s)["pred"].is_file()]
if missing:
    raise FileNotFoundError(
        "No predictions yet for "
        + ", ".join(missing)
        + ". Run: bash code/External_val/BRCA/demo_uni2_A2.sh all"
    )

comp = composition_table(samples)

for head in ("L1", "L12", "L2"):
    sub = comp.loc[comp["head"] == head].copy()
    order = sub.groupby("class")["fraction"].max().sort_values(ascending=False).index
    wide = (
        sub.pivot_table(index="class", columns="sample", values=["fraction", "n", "mean_conf"])
        .reindex(order)
    )
    print(f"\n===== {head} =====")
    display(wide.round(3))


In [ ]:
import matplotlib.pyplot as plt

for fig in plot_spatial(
    samples,
    figsize=FIGSIZE,
    dpi=DPI,
    he_max_side=HE_MAX_SIDE,
    marker_px=MARKER_PX,
    save=SAVE_FIGURES,
    save_dpi=SAVE_DPI,
):
    display(fig)
    plt.close(fig)


### plot the ground truth of 10x dataset:
- data path: /nobackup2/users/lingyu/Python/Hist2Pheno/data/Visium/BRCA/10x_data

In [ ]:
## plot V1_Breast_Cancer_Block_A_Section_1_image
## Spot-level pathologist ground truth from metadata.tsv (fine_annot_type).
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

SAMPLE = "V1_Breast_Cancer_Block_A_Section_1_image"
META_PATH = REPO / "data/Visium/BRCA/10x_data/metadata.tsv"
he_path = sample_paths(SAMPLE)["he"]
pos_path = sample_paths(SAMPLE)["tissue_positions"]

FINE_ORDER = [
    "Healthy_1", "Healthy_2",
    "IDC_1", "IDC_2", "IDC_3", "IDC_4", "IDC_5", "IDC_6", "IDC_7",
    "DCIS/LCIS_1", "DCIS/LCIS_2", "DCIS/LCIS_3", "DCIS/LCIS_4", "DCIS/LCIS_5",
    "Tumor_edge_1", "Tumor_edge_2", "Tumor_edge_3", "Tumor_edge_4", "Tumor_edge_5", "Tumor_edge_6",
]
FINE_COLORS = {
    "Healthy_1": "#1b9e77", "Healthy_2": "#66c2a5",
    "IDC_1": "#7f0000", "IDC_2": "#b2182b", "IDC_3": "#d6604d",
    "IDC_4": "#f4a582", "IDC_5": "#ef6c00", "IDC_6": "#e65100", "IDC_7": "#ffab00",
    "DCIS/LCIS_1": "#4a148c", "DCIS/LCIS_2": "#6a1b9a", "DCIS/LCIS_3": "#8e24aa",
    "DCIS/LCIS_4": "#ab47bc", "DCIS/LCIS_5": "#ce93d8",
    "Tumor_edge_1": "#01579b", "Tumor_edge_2": "#0277bd", "Tumor_edge_3": "#0288d1",
    "Tumor_edge_4": "#039be5", "Tumor_edge_5": "#4fc3f7", "Tumor_edge_6": "#81d4fa",
}

meta = pd.read_csv(META_PATH, sep="\t")
pos = pd.read_csv(
    pos_path, header=None,
    names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row_in_fullres", "pxl_col_in_fullres"],
)
spots = meta.merge(pos, left_on="ID", right_on="barcode", how="left")
if spots["barcode"].isna().any():
    raise KeyError(f"{int(spots['barcode'].isna().sum())} metadata barcodes are missing from {pos_path.name}")
missing_color = sorted(set(spots["fine_annot_type"]) - set(FINE_COLORS))
if missing_color:
    raise KeyError(f"fine_annot_type without a color: {missing_color}")

with tifffile.TiffFile(he_path) as tif:
    height, width = tif.pages[0].shape[:2]
step = max(1, int(np.ceil(max(height, width) / 1600)))
image = np.asarray(tifffile.memmap(he_path)[::step, ::step])
n_r, n_c = image.shape[:2]
extent = (-0.5 * step, (n_c - 0.5) * step, (n_r - 0.5) * step, -0.5 * step)

fig, ax = plt.subplots(figsize=(11.4/2, 8.6/2), dpi=DPI, facecolor="white")
ax.imshow(image, extent=extent, interpolation="bilinear")
# Marker diameter tracks the Visium spot (~177 fullres px) on this axis.
span = float(spots["pxl_col_in_fullres"].max() - spots["pxl_col_in_fullres"].min())
spot_px = 177.48
ax_width_pt = 7.6 * 72
marker_pt = spot_px / span * ax_width_pt
for name in FINE_ORDER:
    part = spots.loc[spots["fine_annot_type"] == name]
    if part.empty:
        continue
    ax.scatter(
        part["pxl_col_in_fullres"],
        part["pxl_row_in_fullres"],
        s=marker_pt ** 2/3,
        c=FINE_COLORS[name],
        linewidths=0,
        rasterized=True,
        zorder=3,
    )
pad = 800
ax.set_xlim(spots["pxl_col_in_fullres"].min() - pad, spots["pxl_col_in_fullres"].max() + pad)
ax.set_ylim(spots["pxl_row_in_fullres"].max() + pad, spots["pxl_row_in_fullres"].min() - pad)
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title(
    f"{SAMPLE}\nspot ground truth  n={len(spots):,}  (metadata.tsv fine_annot_type)",
    fontsize=11,
)
handles = [
    Line2D([0], [0], linestyle="none", marker="o", markersize=7,
           markerfacecolor=FINE_COLORS[name], markeredgecolor="none", label=name)
    for name in FINE_ORDER if (spots["fine_annot_type"] == name).any()
]
ax.legend(
    handles=handles, loc="center left", bbox_to_anchor=(1.01, 0.5),
    frameon=False, fontsize=8, title="fine_annot_type", title_fontsize=9,
    handletextpad=0.4, labelspacing=0.28,
)
fig.tight_layout()

SAVE_FIGURES = True

if SAVE_FIGURES:
    out = sample_paths(SAMPLE)["pred"].parent.parent / "figures" / f"{SAMPLE}_spot_gt_fine_annot.pdf"
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    fig.savefig(out, bbox_inches="tight", facecolor="white")
    print("Saved", out)
plt.show()


In [ ]:
## plot V1_Breast_Cancer_Block_A_Section_1_image
## Same spots as above, merged to 4 labels:
## DCIS/LCIS (5), IDC (7), Tumor edge (6), Healthy (2).
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

SAMPLE = "V1_Breast_Cancer_Block_A_Section_1_image"
META_PATH = REPO / "data/Visium/BRCA/10x_data/metadata.tsv"
he_path = sample_paths(SAMPLE)["he"]
pos_path = sample_paths(SAMPLE)["tissue_positions"]

COARSE_ORDER = ["DCIS/LCIS", "IDC", "Tumor edge", "Healthy"]
COARSE_COLORS = {
    "DCIS/LCIS": "#6a1b9a",
    "IDC": "#d6604d",
    "Tumor edge": "#0277bd",
    "Healthy": "#1b9e77",
}

def coarse_label(fine: str) -> str:
    name = str(fine)
    if name.startswith("DCIS/LCIS"):
        return "DCIS/LCIS"
    if name.startswith("IDC"):
        return "IDC"
    if name.startswith("Tumor_edge"):
        return "Tumor edge"
    if name.startswith("Healthy"):
        return "Healthy"
    raise KeyError(f"unmapped fine_annot_type: {name}")

meta = pd.read_csv(META_PATH, sep="\t")
pos = pd.read_csv(
    pos_path, header=None,
    names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row_in_fullres", "pxl_col_in_fullres"],
)
spots = meta.merge(pos, left_on="ID", right_on="barcode", how="left")
if spots["barcode"].isna().any():
    raise KeyError(f"{int(spots['barcode'].isna().sum())} metadata barcodes are missing from {pos_path.name}")
spots["label"] = spots["fine_annot_type"].map(coarse_label)
print(spots.groupby("label")["fine_annot_type"].nunique().reindex(COARSE_ORDER).to_string())
print(spots["label"].value_counts().reindex(COARSE_ORDER).to_string())

with tifffile.TiffFile(he_path) as tif:
    height, width = tif.pages[0].shape[:2]
step = max(1, int(np.ceil(max(height, width) / 1600)))
image = np.asarray(tifffile.memmap(he_path)[::step, ::step])
n_r, n_c = image.shape[:2]
extent = (-0.5 * step, (n_c - 0.5) * step, (n_r - 0.5) * step, -0.5 * step)

fig, ax = plt.subplots(figsize=(11.4/2, 8.6/2), dpi=DPI, facecolor="white")
ax.imshow(image, extent=extent, interpolation="bilinear")
span = float(spots["pxl_col_in_fullres"].max() - spots["pxl_col_in_fullres"].min())
spot_px = 177.48
ax_width_pt = 7.6 * 72
marker_pt = spot_px / span * ax_width_pt
for name in COARSE_ORDER:
    part = spots.loc[spots["label"] == name]
    if part.empty:
        continue
    ax.scatter(
        part["pxl_col_in_fullres"],
        part["pxl_row_in_fullres"],
        s=marker_pt ** 2/3,
        c=COARSE_COLORS[name],
        linewidths=0,
        rasterized=True,
        zorder=3,
    )
pad = 800
ax.set_xlim(spots["pxl_col_in_fullres"].min() - pad, spots["pxl_col_in_fullres"].max() + pad)
ax.set_ylim(spots["pxl_row_in_fullres"].max() + pad, spots["pxl_row_in_fullres"].min() - pad)
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title(
    f"{SAMPLE}\nspot ground truth  n={len(spots):,}  (4 merged labels)",
    fontsize=11,
)
handles = [
    Line2D([0], [0], linestyle="none", marker="o", markersize=7,
           markerfacecolor=COARSE_COLORS[name], markeredgecolor="none", label=name)
    for name in COARSE_ORDER if (spots["label"] == name).any()
]
ax.legend(
    handles=handles, loc="center left", bbox_to_anchor=(1.01, 0.5),
    frameon=False, fontsize=9, title="label", title_fontsize=9,
    handletextpad=0.4, labelspacing=0.35,
)
fig.tight_layout()
if SAVE_FIGURES:
    out = sample_paths(SAMPLE)["pred"].parent.parent / "figures" / f"{SAMPLE}_spot_gt_4class.pdf"
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    fig.savefig(out, bbox_inches="tight", facecolor="white")
    print("Saved", out)
plt.show()


In [ ]:
## plot V1_Breast_Cancer_Block_A_Section_1_image
## Hist2Pheno L2: one spatial panel per predicted cell type.
import numpy as np
import anndata as ad
import tifffile
import matplotlib.pyplot as plt

SAMPLE = "V1_Breast_Cancer_Block_A_Section_1_image"
he_path = sample_paths(SAMPLE)["he"]
pred_path = sample_paths(SAMPLE)["pred"]
adata = ad.read_h5ad(pred_path)
xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
labels = adata.obs["pred_CT"].astype(str).to_numpy()
palette = brca_ext.COHORT._palette()["L2"]
counts = adata.obs["pred_CT"].astype(str).value_counts()
names = [n for n in palette if n in counts.index]
names += [n for n in counts.index if n not in set(names)]
print(f"{SAMPLE}  nuclei={len(xy):,}  L2 types={len(names)}")
print(counts.reindex(names).to_string())

with tifffile.TiffFile(he_path) as tif:
    height, width = tif.pages[0].shape[:2]
step = max(1, int(np.ceil(max(height, width) / 1600)))
image = np.asarray(tifffile.memmap(he_path)[::step, ::step])
n_r, n_c = image.shape[:2]
extent = (-0.5 * step, (n_c - 0.5) * step, (n_r - 0.5) * step, -0.5 * step)

lo = np.percentile(xy, 0.5, axis=0)
hi = np.percentile(xy, 99.5, axis=0)
pad = 400
xlim = (float(lo[0]) - pad, float(hi[0]) + pad)
ylim = (float(hi[1]) + pad, float(lo[1]) - pad)

ncol = 5
nrow = int(np.ceil(len(names) / ncol))
fig, axes = plt.subplots(
    nrow, ncol,
    figsize=(ncol * 3.35, nrow * 3.15),
    dpi=DPI,
    facecolor="white",
)
axes = np.atleast_1d(axes).ravel()
marker_px = 2.2
point_s = (marker_px * 72.0 / DPI) ** 2
for ax, name in zip(axes, names):
    ax.imshow(image, extent=extent, interpolation="bilinear")
    mask = labels == name
    color = palette.get(name, (0.55, 0.55, 0.55, 1.0))
    ax.scatter(
        xy[mask, 0],
        xy[mask, 1],
        s=point_s,
        c=[color],
        linewidths=0,
        rasterized=True,
        zorder=3,
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(f"{name}  n={int(mask.sum()):,}", fontsize=8, color="black")
for ax in axes[len(names):]:
    ax.set_axis_off()
fig.suptitle(
    f"{SAMPLE}\nHist2Pheno L2 cell type  n={len(xy):,}",
    fontsize=12,
    y=1.01,
)
fig.tight_layout()
if SAVE_FIGURES:
    out = pred_path.parent.parent / "figures" / f"{SAMPLE}_hist2pheno_L2_per_class.pdf"
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    fig.savefig(out, bbox_inches="tight", facecolor="white")
    print("Saved", out)
plt.show()


In [ ]:
## plot V1_Breast_Cancer_Block_A_Section_1_image
## Three panels, same layout as the NPC nuclei / pseudo-spot comparison:
## Hist2Pheno L2 nuclei, majority-L2 pseudo-spots, 20-class spot ground truth.
import json

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.spatial import cKDTree

from Hist2Pheno_pkg.nuclei_to_spots import nuclei_to_spots
from Hist2Pheno_pkg.plotting_palettes import get_palette

SAMPLE = "V1_Breast_Cancer_Block_A_Section_1_image"
DATA_10X = REPO / "data/Visium/BRCA/10x_data"
L2_PALETTE = get_palette("pancancer", "fine")
L2_NAMES = list(L2_PALETTE)
FINE_ORDER = [
    "Healthy_1", "Healthy_2",
    "IDC_1", "IDC_2", "IDC_3", "IDC_4", "IDC_5", "IDC_6", "IDC_7",
    "DCIS/LCIS_1", "DCIS/LCIS_2", "DCIS/LCIS_3", "DCIS/LCIS_4", "DCIS/LCIS_5",
    "Tumor_edge_1", "Tumor_edge_2", "Tumor_edge_3", "Tumor_edge_4", "Tumor_edge_5", "Tumor_edge_6",
]
FINE_COLORS = {
    "Healthy_1": "#1b9e77", "Healthy_2": "#66c2a5",
    "IDC_1": "#7f0000", "IDC_2": "#b2182b", "IDC_3": "#d6604d",
    "IDC_4": "#f4a582", "IDC_5": "#ef6c00", "IDC_6": "#e65100", "IDC_7": "#ffab00",
    "DCIS/LCIS_1": "#4a148c", "DCIS/LCIS_2": "#6a1b9a", "DCIS/LCIS_3": "#8e24aa",
    "DCIS/LCIS_4": "#ab47bc", "DCIS/LCIS_5": "#ce93d8",
    "Tumor_edge_1": "#01579b", "Tumor_edge_2": "#0277bd", "Tumor_edge_3": "#0288d1",
    "Tumor_edge_4": "#039be5", "Tumor_edge_5": "#4fc3f7", "Tumor_edge_6": "#81d4fa",
}

def legend_handles(names, palette, markersize=8):
    return [
        Line2D(
            [0], [0], marker="o", color="none",
            markerfacecolor=palette[name][:3] if not isinstance(palette[name], str) else palette[name],
            markeredgecolor="none", markersize=markersize, label=name,
        )
        for name in names
        if name in palette
    ]

def draw_spot_circles(ax, xy, labels, radius, palette, order):
    patches, colors = [], []
    labels = np.asarray(labels)
    for name in order:
        hit = labels == name
        if not np.any(hit):
            continue
        col = palette[name]
        rgb = col[:3] if not isinstance(col, str) else col
        for x, y in np.asarray(xy)[hit]:
            patches.append(Circle((x, y), radius))
            colors.append(rgb)
    if patches:
        ax.add_collection(PatchCollection(patches, facecolors=colors, edgecolors="none"))

scale = json.loads((DATA_10X / "spatial/scalefactors_json.json").read_text())
radius_px = float(scale["spot_diameter_fullres"]) / 2
pos = pd.read_csv(
    DATA_10X / "spatial/tissue_positions_list.csv",
    header=None,
    names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"],
)
pos = pos.loc[pos["in_tissue"] == 1].reset_index(drop=True)
pos["barcode"] = pos["barcode"].astype(str)
spot_xy = pos[["pxl_col", "pxl_row"]].to_numpy(dtype=np.float64)
pitch_px = float(np.median(cKDTree(spot_xy).query(spot_xy, k=2)[0][:, 1]))

meta = pd.read_csv(DATA_10X / "metadata.tsv", sep="\t")
gt = pos.merge(meta, left_on="barcode", right_on="ID", how="inner")
if len(gt) != len(pos):
    raise KeyError(f"ground truth matched {len(gt):,} / {len(pos):,} in-tissue spots")
missing_color = sorted(set(gt["fine_annot_type"]) - set(FINE_COLORS))
if missing_color:
    raise KeyError(f"fine_annot_type without a color: {missing_color}")
gt_xy = gt[["pxl_col", "pxl_row"]].to_numpy(dtype=np.float64)
gt_lab = gt["fine_annot_type"].astype(str).to_numpy()

pred_path = sample_paths(SAMPLE)["pred"]
adata = ad.read_h5ad(pred_path)
nuc_xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
nuc_lab = adata.obs["pred_CT"].astype(str).to_numpy()
captured = nuclei_to_spots(
    nuc_xy, spot_xy, radius_px, shape="circle",
    labels=nuc_lab, class_names=L2_NAMES,
)
used = captured["n_nuclei"] > 0
dominant = captured["dominant"][used]
print(
    f"{SAMPLE}: nuclei {len(nuc_xy):,}, in-tissue spots {len(spot_xy):,}, "
    f"pitch {pitch_px:.1f} px, capture radius {radius_px:.1f} px, "
    f"nuclei in a spot {int(captured['inside'].sum()):,} / {len(nuc_xy):,}, "
    f"spots with nuclei {int(used.sum()):,}"
)

nucleus_s = max(0.12, (float(MARKER_PX) * 72.0 / float(DPI)) ** 2 * 0.35)
spot_draw_radius = 0.50 * pitch_px
fig, axes = plt.subplots(1, 3, figsize=(16.8, 6.2), dpi=DPI, facecolor="white")
axes[0].set_title("Hist2Pheno L2")
for name in L2_NAMES:
    mask = nuc_lab == name
    if np.any(mask):
        axes[0].scatter(
            nuc_xy[mask, 0], nuc_xy[mask, 1], s=nucleus_s,
            c=[L2_PALETTE[name]], linewidths=0, rasterized=True,
        )
draw_spot_circles(
    axes[1], spot_xy[used], dominant,
    spot_draw_radius, L2_PALETTE, L2_NAMES,
)
axes[1].set_title("Hist2Pheno_spot")
draw_spot_circles(
    axes[2], gt_xy, gt_lab,
    spot_draw_radius, FINE_COLORS, FINE_ORDER,
)
axes[2].set_title("Ground truth_20")

pad = pitch_px
for ax in axes:
    ax.set_xlim(nuc_xy[:, 0].min() - pad, nuc_xy[:, 0].max() + pad)
    ax.set_ylim(nuc_xy[:, 1].max() + pad, nuc_xy[:, 1].min() - pad)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

present_l2 = [n for n in L2_NAMES if n in set(nuc_lab) or n in set(dominant)]
axes[1].legend(
    handles=legend_handles(present_l2, L2_PALETTE),
    frameon=False, fontsize=8, loc="upper center",
    bbox_to_anchor=(0.5, -0.02), ncol=4,
)
axes[2].legend(
    handles=legend_handles(FINE_ORDER, FINE_COLORS, markersize=7),
    frameon=False, fontsize=7, loc="upper center",
    bbox_to_anchor=(0.5, -0.02), ncol=2,
)
fig.suptitle(
    f"{SAMPLE}  Hist2Pheno L2 / Hist2Pheno_spot / Ground truth_20",
    fontsize=12, y=1.02,
)
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
if SAVE_FIGURES:
    out = pred_path.parent.parent / "figures" / f"{SAMPLE}_L2_spot_gt20.pdf"
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    fig.savefig(out, bbox_inches="tight", facecolor="white")
    print("Saved", out)
plt.show()


### Single section + QuPath ROI

Section: `200219_BC_SA3_V19T26-031_CE_V4-Spot000001`  
ROI: `data/Visium/BRCA/HE_images/selected_rectangle_coordinates.tsv` (native HE pixels).


In [ ]:
# One section only (full slide L1 / L12 / L2)
SAMPLE_ROI = "200219_BC_SA3_V19T26-031_CE_V4-Spot000001"
if SAMPLE_ROI not in samples:
    raise FileNotFoundError(
        f"{SAMPLE_ROI} not in discover_samples(): {samples}. "
        "Run: SAMPLE=... bash code/External_val/BRCA/demo_uni2_A2.sh all"
    )
if not sample_paths(SAMPLE_ROI)["pred"].is_file():
    raise FileNotFoundError(sample_paths(SAMPLE_ROI)["pred"])

for fig in plot_spatial(
    [SAMPLE_ROI],
    figsize=FIGSIZE,
    dpi=DPI,
    he_max_side=HE_MAX_SIDE,
    marker_px=MARKER_PX,
    save=SAVE_FIGURES,
    save_dpi=SAVE_DPI,
):
    display(fig)
    plt.close(fig)


In [ ]:
# ROI from QuPath rectangle → L1 / L12 / L2 on cropped HE
# Full-slide MARKER_PX (~1) is too small once the view is a ~2.5k×2k crop.
ROI_TSV = REPO / "data" / "Visium" / "BRCA" / "HE_images" / "selected_rectangle_coordinates.tsv"
MARKER_PX_ROI = max(float(MARKER_PX) * 5.0, 5.0)
fig_h = float(FIGSIZE[1]) * 1.9 if FIGSIZE[1] <= 8.5 else float(FIGSIZE[1])

fig, pred, in_roi, roi = plot_roi_spatial(
    SAMPLE_ROI,
    ROI_TSV,
    marker_px=MARKER_PX_ROI,
    dpi=DPI,
    figsize=(float(FIGSIZE[0]), fig_h),
    legend_markersize=13,
    legend_fontsize=13,
    title_fontsize=11,
    save=SAVE_FIGURES,
    save_dpi=SAVE_DPI,
)
print(
    f"ROI {SAMPLE_ROI}: x=[{roi['x_min_px']:.0f},{roi['x_max_px']:.0f}] "
    f"y=[{roi['y_min_px']:.0f},{roi['y_max_px']:.0f}]  "
    f"nuclei={int(in_roi.sum()):,}  MARKER_PX_ROI={MARKER_PX_ROI:.1f}"
)
display(fig)
plt.close(fig)


In [ ]:
# ROI cell-type composition (100% stacked): L1 / L12 / L2
fig, comp_df = plot_roi_composition(
    SAMPLE_ROI,
    pred=pred,
    in_roi=in_roi,
    dpi=DPI,
    figsize=(12.0, 4.2),
    legend_fontsize_l1=9,
    legend_fontsize_l2=8,
    title_fontsize=11,
    save=SAVE_FIGURES,
    save_dpi=SAVE_DPI,
)
display(comp_df.pivot_table(index="class", columns="head", values="fraction").fillna(0).round(3))
display(fig)
plt.close(fig)


In [ ]:
for fig in plot_composition(samples, save=SAVE_FIGURES, dpi=DPI, save_dpi=SAVE_DPI):
    display(fig)
    plt.close(fig)


## Histology-derived niche indices (TLS / SRI / TNI)

Same implementation as `code/Xenium_lung/histology_derived_niche_index.ipynb` §3, via `Hist2Pheno_pkg/histology_niche_index.py`. Lung ARI / FRI are not used: those members (AT1/AT2, activated fibrotic FBs, SPP1+ macrophages) are not pan-cancer L2 classes.

Scores use **L2 softmax**, not the argmax label. `ensure_l2_probs` caches `{sample}_l2_probs.npz` (first call reloads the cross-section checkpoint).

| Index | Abundance | Spatial (radius R) |
|--|--|--|
| **TLS** | sum(B_cell + Plasma + CD4_T + CD8_T + Tregs + DC) | `(B_local × T_local × DC_local)^(1/3)` |
| **SRI** | (Fibroblast + Myofibroblast + Stromal) / epithelium | Fib_local / Epi_local |
| **TNI** | (fibroblast + Macrophage + Endothelial) / epithelium | `(Fib × Mac × Endo)^(1/3) / Epi_local` |

Epithelium = Epithelial, Tumor, Alveolar, Injury_epithelial, DCIS, Myoepithelial. NK is left out of TLS, as in the lung T-cell set. `spatial_HE` is native HE pixels, so R = **55 µm** is converted with **each sample's** `geometry(sample)['mpp_um']` (GSM ~0.548; Zenodo JPEGs ~0.172), not 0.5. Hotspots are the within-slide 95th percentile. TLS, SRI, and TNI each use a 1×3 row: abundance, the spatial score without markers, then the same spatial map with cyan candidate nuclei. The two spatial panels share one color scale.


In [ ]:
import importlib

for _p in (REPO / "code" / "Hist2Pheno_pkg", REPO / "code" / "Xenium_lung"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import histology_niche_index as hni
importlib.reload(hni)
import histology_derived_niche_index as hdni
importlib.reload(hdni)
from histology_niche_index import OrganNicheConfig

import anndata as ad

TLS_RADIUS_UM = 55.0  # lung notebook cross-dataset TLS radius
RECOMPUTE_NICHE = True  # 2026.10.28, legacy niche cache lacks provenance; recompute from verified current probabilities

BRCA_NICHE = OrganNicheConfig(
    pan_organ="pancancer_brca",
    tls_b=("B_cell", "Plasma"),
    tls_t=("CD4_T", "CD8_T", "Tregs"),
    tls_dc=("DC",),
    epithelium=(
        "Epithelial", "Tumor", "Alveolar",
        "Injury_epithelial", "DCIS", "Myoepithelial",
    ),
    fibroblast=("Fibroblast", "Myofibroblast", "Stromal"),
    endothelial=("Endothelial",),
    macrophage=("Macrophage",),
    um_per_he_pixel=float(geometry(samples[0])["mpp_um"]),  # formula table only; cell below uses per-sample mpp
    spatial_radius_um=TLS_RADIUS_UM,
    cohort="BRCA",
)
print("µm/px by sample:", {s: round(float(geometry(s)["mpp_um"]), 4) for s in samples})
display(hni.index_formula_table(BRCA_NICHE))


In [ ]:
import numpy as np
niche_frames = {}
for sample in samples:
    paths = sample_paths(sample)
    niche_path = paths["pred"].with_name(f"{sample}_niche_indices.h5ad")
    mpp = float(geometry(sample)["mpp_um"])
    if niche_path.is_file() and not RECOMPUTE_NICHE:
        cached = ad.read_h5ad(niche_path)
        df = cached.obs.copy()
        xy = np.asarray(cached.obsm["spatial_HE"], dtype=np.float64)
        df["coord_x"] = xy[:, 0]
        df["coord_y"] = xy[:, 1]
        print(f"[niche] cache {niche_path.name}  n={len(df):,}  µm/px={mpp:.4f}")
    else:
        probs, class_names = ensure_l2_probs(sample)
        missing = hni.missing_index_cell_types(class_names, BRCA_NICHE)
        if missing:
            raise RuntimeError(f"{sample} missing niche compartments: {missing}")
        pred = ad.read_h5ad(paths["pred"])
        coords = np.asarray(pred.obsm["spatial_HE"], dtype=np.float64)
        df = pred.obs.copy()
        df["coord_x"] = coords[:, 0]
        df["coord_y"] = coords[:, 1]
        df = hni.add_abundance_indices(df, probs, class_names, BRCA_NICHE)
        df = hni.add_spatial_indices(
            df, probs, class_names, coords, BRCA_NICHE,
            radius_um=TLS_RADIUS_UM,
            um_per_pixel=mpp,
        )
        out = ad.AnnData(obs=df.drop(columns=["coord_x", "coord_y"]))
        out.obsm["spatial_HE"] = coords
        out.uns["tls_radius_um"] = TLS_RADIUS_UM
        out.uns["um_per_px"] = mpp
        __import__("pipeline_provenance").publish_derived_h5ad(out,niche_path,paths["pred"], probabilities=probs, class_names=class_names, obs_names=pred.obs_names, parameters={"config":BRCA_NICHE,"mpp":mpp,"radius_um":TLS_RADIUS_UM},algorithm=hni.__file__)
        print(f"[niche] wrote {niche_path}  n={len(df):,}")
    niche_frames[sample] = df
    print(f"  TLS hotspot: {hni.summarize_tls_hotspots(df)}")
    print(
        f"  mean TLS={df['idx_TLS'].mean():.3f}  TLS_spatial={df['idx_TLS_spatial'].mean():.3f}  "
        f"SRI={df['idx_SRI_ratio'].mean():.3f}  TNI={df['idx_TNI_ratio'].mean():.3f}"
    )

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

fig_dir = PRED_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)


def tls_score_row(df, sample: str) -> dict:
    """Quantify abundance TLS vs spatial TLS on the same nuclei."""
    a = df["idx_TLS"].to_numpy(dtype=float)
    s = df["idx_TLS_spatial"].to_numpy(dtype=float)
    a_hi = a >= np.percentile(a, 95)
    s_hi = df["tls_candidate"].fillna(False).astype(bool).to_numpy()
    inter = int(np.logical_and(a_hi, s_hi).sum())
    union = int(np.logical_or(a_hi, s_hi).sum())
    n_a = int(a_hi.sum())
    return {
        "sample": sample,
        "n": int(len(df)),
        "TLS_mean": float(np.mean(a)),
        "TLS_median": float(np.median(a)),
        "TLS_p95": float(np.percentile(a, 95)),
        "TLS_max": float(np.max(a)),
        "TLS_spatial_mean": float(np.mean(s)),
        "TLS_spatial_median": float(np.median(s)),
        "TLS_spatial_p95": float(np.percentile(s, 95)),
        "TLS_spatial_max": float(np.max(s)),
        "pearson": float(np.corrcoef(a, s)[0, 1]),
        "spearman": float(pd.Series(a).corr(pd.Series(s), method="spearman")),
        "top5_jaccard": (inter / union) if union else float("nan"),
        "abundance_top5_also_spatial_top5": (inter / n_a) if n_a else float("nan"),
        "TLS_spatial_mean_in_abundance_top5": float(s[a_hi].mean()) if n_a else float("nan"),
        "TLS_mean_in_spatial_top5": float(a[s_hi].mean()) if s_hi.any() else float("nan"),
    }


quant_rows = [tls_score_row(df, sample) for sample, df in niche_frames.items()]
tls_quant = pd.DataFrame(quant_rows)
tls_quant_path = fig_dir / "tls_score_summary.csv"
tls_quant.to_csv(tls_quant_path, index=False)
print(tls_quant_path)
display(tls_quant.round(3))

for sample, df in niche_frames.items():
    fig, axes = hdni.plot_tls_abundance_vs_spatial(
        df,
        sample,
        radius_um=TLS_RADIUS_UM,
        use_entropy=False,
        split_spatial_candidates=True,
        dpi=DPI,
        save_dpi=SAVE_DPI,
        point_size=0.15,
        candidate_size=1.2,
        save_path=(fig_dir / f"{sample}_TLS_abundance_vs_spatial.png") if SAVE_FIGURES else None,
    )
    display(fig)
    plt.close(fig)

    for abundance_col, spatial_col, candidate_col, tag in (
        ("idx_SRI_ratio", "idx_SRI_spatial", "sri_candidate", "SRI"),
        ("idx_TNI_ratio", "idx_TNI_spatial", "tni_candidate", "TNI"),
    ):
        fig, axes = hdni.plot_tls_abundance_vs_spatial(
            df,
            sample,
            radius_um=TLS_RADIUS_UM,
            abundance_col=abundance_col,
            spatial_col=spatial_col,
            candidate_col=candidate_col,
            score_name=tag,
            use_entropy=False,
            split_spatial_candidates=True,
            dpi=DPI,
        save_dpi=SAVE_DPI,
        point_size=0.15,
            candidate_size=1.2,
            save_path=(fig_dir / f"{sample}_{tag}_abundance_vs_spatial.png") if SAVE_FIGURES else None,
        )
        display(fig)
        plt.close(fig)
